In [ ]:
import pandas as pd

In [ ]:

df=pd.read_csv("IMDB Dataset.csv")

In [ ]:
df.drop_duplicates(inplace=True)

In [ ]:
df["review"]=df["review"].str.lower()



In [ ]:
import re
def url_remove(text):
 text=re.sub(r"http\S+","",text)
 return text
df["reviews"]=df["review"].apply(url_remove)

In [ ]:
def remove_punct(text):
  text=re.sub(r"[^A-Za-z0-9\s]","",text)

  return text

df["reviews"]=df["review"].apply(remove_punct)

In [ ]:
def remove_html(text):
  text=re.sub(r"<.*?>","",text)

  return text

df["reviews"]=df["review"].apply(remove_html)

In [ ]:
import nltk

nltk.download("punkt")
nltk.download("punk_teb")
nltk.download("stopwords")

In [ ]:
from nltk.tokenize import  word_tokenize
from nltk.corpus import stopwords




In [ ]:
def remove_stopwords(text):
  tokens=word_tokenize(text)
  stop_words=stopwords.words("english")

  for word in tokens:
    if word in stop_words:
      text=text.replace(word,"")
    return text

df["reviews"]=df["review"].apply(remove_stopwords)



In [ ]:
from nltk.stem import PorterStemmer

def stemming(text):
  PS=PorterStemmer()
  stemmed_words=[]
  tokens=word_tokenize(text)
  for word in tokens:
    stemmed_token=PS.stem(word)
    stemmed_words.append(stemmed_token)
   return "".join(stemmed_words)
df["reviews"]=df["review"].apply(stemming)


In [ ]:
from sklearn.preprocessing import LabelEncoder

le=LabelEncoder()
df["sentiment"=le.fit_transform(df["sentiment"])]


y=df["sentiment"]


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

  tf =TfidfVectorizer(max_features=500)
  X=tf.fit_transform(df["review"])



In [ ]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test=train_test_split(
    X,y,test_size=0.2,random_state=42
)



In [ ]:
import torch

from torch.utils.data import Dataset,DataLoader



In [ ]:
X_train=X_train.toarray()
X_test=X_test.toarray()

In [ ]:
train_set=TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values).float()
)

test_set=TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float()
)



In [ ]:
train_loader=DataLoader(train_set,shuffle=True,batch_size=64)

test_loader=DataLoader(test_set,shuffle=True,batch_size=64)


In [ ]:
import torch.nn as nn
import torch.optim as optimizer

In [ ]:
class RNN(nn.Module):
    def __init__(self,input_size,hidden_size=128,num_layers=1):
      super().__init__()
      self.hidden_size=hidden_size
      self.num_layers=num_layers

      self.rnn=nn.RNN(input_size,hidden_size,num_layers,batch_first=True)

      self.fc=nn.Linear(hidden_size,1)

   def forward(self,x):
     h0=torch.zeros(self.num_layers,x.size(0),self.hidden_size)
      out,_=self.rnn(x,h0)
      out=self.fc(out[:-1,:])
      return out


In [ ]:

input_size=X_train.shape[1]
model=RNN(input_size)

criterion = nn.BCEWithLogitsLoss()
optimizer=optim.Adam(model.parameters())


In [ ]:
epoch=10
for epoch in range(epochs):
  model.train()
  for Xb,yb in train_loader:
    optimizer.zero_grad()
    Xb=Xb.unsqueeze(1)
    outputs = model(Xb).squeeze()

    loss = criterion(outputs, yb)
    loss.backward()
    optimizer.step()
  print(f"{epoch}/{epochs} and loss ={loss.item()}")

